# Shinhan Card 파일 불러오기

## 0. 초기 세팅
각종 라이브러리를 불러오고 초기값 등을 세팅합니다.

In [1]:
#!uv add xlrd
#!uv add lxml

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
from ledgerly.expenditure import (shinhan_file_config
                                  , map_shinhan_card_df_to_expenditure
                                  , preprocess_shinhan_data
                                  , map_category
                                  , insert_expenditure_data
                                  , fetch_expenditure_data)


## 1. 데이터 파일 조회 및 전처리

In [6]:
sh_ori_df = pd.read_excel(shinhan_file_config["shinhan_xls"])
len(sh_ori_df)

9

In [7]:
sh_df = preprocess_shinhan_data(sh_ori_df)
sh_df.head()

,거래일,카드구분,이용카드,가맹점명,승인번호,금액,매입구분,이용구분,거래통화,해외이용금액,취소상태
0,2026-01-30 16:17:00,신용,본인204*,홈플러스강서점,8245718,5160,결제확정,일시불,NaN,NaN,NaN
1,2026-01-24 11:08:00,신용,본인204*,홈플러스㈜,25863165,68210,결제확정,일시불,NaN,NaN,NaN
2,2026-01-16 21:55:00,신용,본인204*,홈플러스㈜,31299218,50240,결제확정,일시불,NaN,NaN,NaN
3,2026-01-16 12:35:00,신용,본인204*,(주)아트박스 강서 홈플러스점,23635058,2000,결제확정,일시불,NaN,NaN,NaN
4,2026-01-07 16:48:00,신용,본인204*,새서울약국,8159190,5700,결제확정,일시불,NaN,NaN,NaN


## 2. DB 데이터로 매핑

In [8]:
expenditure_df = map_shinhan_card_df_to_expenditure(sh_df)
expenditure_df.head()

,used_at,payment_type,payment_provider,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid
0,2026-01-30 16:17:00,card,shinhan,홈플러스강서점,single,5160,0,unknown,None,card_shinhan_8245718
1,2026-01-24 11:08:00,card,shinhan,홈플러스㈜,single,68210,0,unknown,None,card_shinhan_25863165
2,2026-01-16 21:55:00,card,shinhan,홈플러스㈜,single,50240,0,unknown,None,card_shinhan_31299218
3,2026-01-16 12:35:00,card,shinhan,(주)아트박스 강서 홈플러스점,single,2000,0,unknown,None,card_shinhan_23635058
4,2026-01-07 16:48:00,card,shinhan,새서울약국,single,5700,0,unknown,None,card_shinhan_8159190


In [9]:
expenditure_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   used_at           8 non-null      datetime64[ns]
 1   payment_type      8 non-null      object        
 2   payment_provider  8 non-null      object        
 3   merchant_name     8 non-null      string        
 4   installment_type  8 non-null      object        
 5   amount            8 non-null      int64         
 6   remaining_amount  8 non-null      int64         
 7   category          8 non-null      object        
 8   memo              0 non-null      object        
 9   source_uid        8 non-null      object        
dtypes: datetime64[ns](1), int64(2), object(6), string(1)
memory usage: 772.0+ bytes


## 3. 지출 분류

In [10]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df["category"] = expenditure_df["category"].fillna("미분류")

In [11]:
expenditure_df[["merchant_name", "category"]].head(20)

,merchant_name,category
0,홈플러스강서점,식비
1,홈플러스㈜,식비
2,홈플러스㈜,식비
3,(주)아트박스 강서 홈플러스점,식비
4,새서울약국,의료
5,코아이비인후과,기타
6,강서홈약국,의료
7,홈플러스㈜,식비


## 4. DB 저장

In [12]:
insert_expenditure_data(expenditure_df)
print(f"{len(expenditure_df)}개의 데이터가 성공적으로 저장되었습니다.")

8개의 데이터가 성공적으로 저장되었습니다.


In [ ]:
fetched_db_df = fetch_expenditure_data()

In [ ]:
fetched_db_df.info()